<a href="https://colab.research.google.com/github/ShahabNaz/ntua-parkinson-dataset/blob/feature/parkinson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class NestedFolderImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        """
        Args:
            root_dir (string): Directory with all the images, possibly nested.
            transform (callable, optional): Optional transform to be applied
                on a sample.
        """
        self.image_paths = []
        self.transform = transform

        # Walk through all subdirectories and collect image file paths
        for dirpath, _, filenames in os.walk(root_dir):
            for filename in filenames:
                if filename.lower().endswith('.png'):
                    self.image_paths.append(os.path.join(dirpath, filename))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load image
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')  # Ensure 3-channel RGB

        # Apply transformations if any
        if self.transform:
            image = self.transform(image)

        return image


In [2]:
!git clone https://github.com/ShahabNaz/ntua-parkinson-dataset.git

Cloning into 'ntua-parkinson-dataset'...
remote: Enumerating objects: 42116, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 42116 (delta 1), reused 3 (delta 0), pack-reused 42104 (from 1)
Receiving objects: 100% (42116/42116), 2.14 GiB | 45.07 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Updating files: 100% (44019/44019), done.


In [3]:
# Define any data transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

path = '/content/ntua-parkinson-dataset/PD_Patients'

# Initialize dataset and dataloader
dataset = NestedFolderImageDataset(root_dir= path, transform=transform)

dataloader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=1)

# Iterate through the dataset
for batch in dataloader:

    pass



In [15]:
dataset

In [4]:
!pip install open_clip_torch==2.23.0 transformers==4.35.2 matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 109.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5

In [10]:
import torch
from urllib.request import urlopen
from PIL import Image
from open_clip import create_model_from_pretrained, get_tokenizer

# Load the model and config files from the Hugging Face Hub
model, preprocess = create_model_from_pretrained('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
tokenizer = get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')


# Zero-shot image classification
template = 'this is a photo of '
labels = [
    'adenocarcinoma histopathology',
    'brain MRI',
    'covid line chart',
    'squamous cell carcinoma histopathology',
    'immunohistochemistry histopathology',
    'bone X-ray',
    'chest X-ray',
    'pie chart',
    'hematoxylin and eosin histopathology'
]


device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
model.eval()

context_length = 256

#images = torch.stack([preprocess(Image.open(urlopen(dataset_url + img))) for img in test_imgs]).to(device)
texts = tokenizer([template + l for l in labels], context_length=context_length).to(device)
img_folder = ['/content/ntua-parkinson-dataset/PD_Patients/Subject10/0.DAT/s1/001.png', '/content/ntua-parkinson-dataset/PD_Patients/Subject10/1.MRI/3D_FL_AX_CYBER_001.png', '/content/ntua-parkinson-dataset/Non PD Patients/Subject1/0.DAT/s1/001.png']

images = torch.stack([preprocess(Image.open(image)) for image in img_folder]).to(device)
with torch.no_grad():
    image_features, text_features, logit_scale = model(images, texts)

    logits = (logit_scale * image_features @ text_features.t()).detach().softmax(dim=-1)
    sorted_indices = torch.argsort(logits, dim=-1, descending=True)

    logits = logits.cpu().numpy()
    sorted_indices = sorted_indices.cpu().numpy()

top_k = -1

for i, img in enumerate(img_folder):
    pred = labels[sorted_indices[i][0]]

    top_k = len(labels) if top_k == -1 else top_k
    print(img.split('/')[-1] + ':')
    for j in range(top_k):
        jth_index = sorted_indices[i][j]
        print(f'{labels[jth_index]}: {logits[i][jth_index]}')
    print('\n')


001.png:
pie chart: 0.999562680721283
brain MRI: 0.0003045571793336421
bone X-ray: 8.491400512866676e-05
hematoxylin and eosin histopathology: 2.8066047889296897e-05
covid line chart: 7.931883374112658e-06
immunohistochemistry histopathology: 5.334160505299224e-06
adenocarcinoma histopathology: 4.703646936832229e-06
chest X-ray: 9.637675475460128e-07
squamous cell carcinoma histopathology: 7.965475674609479e-07


3D_FL_AX_CYBER_001.png:
brain MRI: 0.8521876335144043
hematoxylin and eosin histopathology: 0.06451372057199478
squamous cell carcinoma histopathology: 0.04506748169660568
adenocarcinoma histopathology: 0.02439136803150177
immunohistochemistry histopathology: 0.011580802500247955
bone X-ray: 0.0022373890969902277
chest X-ray: 2.145518192264717e-05
pie chart: 7.223298581493509e-08
covid line chart: 4.2197605409910466e-09


001.png:
pie chart: 0.975075900554657
brain MRI: 0.023253820836544037
hematoxylin and eosin histopathology: 0.0011206802446395159
bone X-ray: 0.0005070046754

# New section